In [1]:
!pip install pandas

  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 4.0 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.9 MB 5.0 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.9 MB 5.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 5.2 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.9 MB 5.2 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.9 MB 5.0 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 5.3 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.9 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 5.8 MB/s  0:00:01
Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl (12.5 MB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (3


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

import sqlite3

# Create an in-memory SQLite database connection

conn = sqlite3.connect(':memory:')

cursor = conn.cursor()

print("Setup complete. Environment ready.")

# Define a messy, unnormalized dataset (0NF)

# Issues: Non-atomic values (multiple text books), redundant data, mixed concerns.

data_0nf_medical = {

    "Patient_ID": [301, 301, 302, 303, 304],

    "Patient_Name": ["John", "John", "Emma", "Noah", "Ava"],

    "Visit_ID": ["V001", "V002", "V003", "V004", "V005"],

    "Doctor_ID": ["D101", "D102", "D101", "D103", "D104"],

    "Doctor_Name": ["Dr. Kumar", "Dr. Ravi", "Dr. Kumar", "Dr. Mehta", "Dr. Singh"],

    "Department": ["General Medicine", "Orthopedics", "General Medicine", "Cardiology", "Dermatology"],

    "Visit_Date": ["2026-01-10", "2026-02-12", "2026-03-05", "2026-03-18", "2026-04-01"],

    "Medications_Prescribed": [

        "Paracetamol, Vitamin C",

        "Painkiller, Calcium",

        "Paracetamol, Antibiotic",

        "BP Tablet, Aspirin",

        "Skin Cream, Antibiotic"

    ]

}

df_0nf = pd.DataFrame(data_0nf_medical)

df_0nf.to_sql('Unnormalized_Leasing', conn, index=False, if_exists='replace')

print("\n--- Unnormalized Data (0NF) ---")

df_0nf

Setup complete. Environment ready.

--- Unnormalized Data (0NF) ---


,Patient_ID,Patient_Name,Visit_ID,Doctor_ID,Doctor_Name,Department,Visit_Date,Medications_Prescribed
0,301,John,V001,D101,Dr. Kumar,General Medicine,2026-01-10,"Paracetamol, Vitamin C"
1,301,John,V002,D102,Dr. Ravi,Orthopedics,2026-02-12,"Painkiller, Calcium"
2,302,Emma,V003,D101,Dr. Kumar,General Medicine,2026-03-05,"Paracetamol, Antibiotic"
3,303,Noah,V004,D103,Dr. Mehta,Cardiology,2026-03-18,"BP Tablet, Aspirin"
4,304,Ava,V005,D104,Dr. Singh,Dermatology,2026-04-01,"Skin Cream, Antibiotic"


In [4]:
# Flattening the Medication_prescribed column to ensure atomicity

df_1nf = df_0nf.assign(Medications_Prescribed=df_0nf['Medications_Prescribed'].str.split(', ')).explode(
    'Medications_Prescribed')

# Save to SQL

df_1nf.to_sql('Table_1NF', conn, index=False, if_exists='replace')

print("--- 1NF Data (Atomic values enforced) ---")

print(f"Row count increased from {len(df_0nf)} to {len(df_1nf)} due to flattening.")

df_1nf

--- 1NF Data (Atomic values enforced) ---
Row count increased from 5 to 10 due to flattening.


,Patient_ID,Patient_Name,Visit_ID,Doctor_ID,Doctor_Name,Department,Visit_Date,Medications_Prescribed
0,301,John,V001,D101,Dr. Kumar,General Medicine,2026-01-10,Paracetamol
0,301,John,V001,D101,Dr. Kumar,General Medicine,2026-01-10,Vitamin C
1,301,John,V002,D102,Dr. Ravi,Orthopedics,2026-02-12,Painkiller
1,301,John,V002,D102,Dr. Ravi,Orthopedics,2026-02-12,Calcium
2,302,Emma,V003,D101,Dr. Kumar,General Medicine,2026-03-05,Paracetamol
2,302,Emma,V003,D101,Dr. Kumar,General Medicine,2026-03-05,Antibiotic
3,303,Noah,V004,D103,Dr. Mehta,Cardiology,2026-03-18,BP Tablet
3,303,Noah,V004,D103,Dr. Mehta,Cardiology,2026-03-18,Aspirin
4,304,Ava,V005,D104,Dr. Singh,Dermatology,2026-04-01,Skin Cream
4,304,Ava,V005,D104,Dr. Singh,Dermatology,2026-04-01,Antibiotic


In [6]:
patient_df = df_1nf[["Patient_ID", "Patient_Name"]].drop_duplicates()

doctor_df = df_1nf[["Doctor_ID", "Doctor_Name", "Department"]].drop_duplicates()

visit_df = df_1nf[["Visit_ID", "Patient_ID", "Doctor_ID", "Visit_Date"]].drop_duplicates()

visit_med_df = df_1nf[["Visit_ID", "Medications_Prescribed"]].drop_duplicates()

print(patient_df, "patient_df")

print(doctor_df, "doctor_df")

print(visit_df, "visit_df")

print(visit_med_df, "visit_med_df")


   Patient_ID Patient_Name
0         301         John
2         302         Emma
3         303         Noah
4         304          Ava patient_df
  Doctor_ID Doctor_Name        Department
0      D101   Dr. Kumar  General Medicine
1      D102    Dr. Ravi       Orthopedics
3      D103   Dr. Mehta        Cardiology
4      D104   Dr. Singh       Dermatology doctor_df
  Visit_ID  Patient_ID Doctor_ID  Visit_Date
0     V001         301      D101  2026-01-10
1     V002         301      D102  2026-02-12
2     V003         302      D101  2026-03-05
3     V004         303      D103  2026-03-18
4     V005         304      D104  2026-04-01 visit_df
  Visit_ID Medications_Prescribed
0     V001            Paracetamol
0     V001              Vitamin C
1     V002             Painkiller
1     V002                Calcium
2     V003            Paracetamol
2     V003             Antibiotic
3     V004              BP Tablet
3     V004                Aspirin
4     V005             Skin Cream
4     V005     

In [7]:
department_df = doctor_df[["Department"]].drop_duplicates().reset_index(drop=True)

# create Dept_ID

department_df["Dept_ID"] = ["DEP" + str(i + 1) for i in range(len(department_df))]

# rearrange columns

department_df = department_df[["Dept_ID", "Department"]]

print(department_df, "department_df")

doctor_df = doctor_df.merge(department_df, on="Department")

doctor_df = doctor_df[["Doctor_ID", "Doctor_Name", "Dept_ID"]]

print(doctor_df, "doctor_df")

medication_df = df_1nf[["Medications_Prescribed"]].drop_duplicates().reset_index(drop=True)

medication_df["Med_ID"] = ["MED" + str(i + 1) for i in range(len(medication_df))]

medication_df = medication_df[["Med_ID", "Medications_Prescribed"]]

print(medication_df, "medication_df")

visit_med_df = visit_med_df.merge(medication_df, on="Medications_Prescribed")

visit_med_df = visit_med_df[["Visit_ID", "Med_ID"]]

print(visit_med_df, "visit_med_df")

  Dept_ID        Department
0    DEP1  General Medicine
1    DEP2       Orthopedics
2    DEP3        Cardiology
3    DEP4       Dermatology department_df
  Doctor_ID Doctor_Name Dept_ID
0      D101   Dr. Kumar    DEP1
1      D102    Dr. Ravi    DEP2
2      D103   Dr. Mehta    DEP3
3      D104   Dr. Singh    DEP4 doctor_df
  Med_ID Medications_Prescribed
0   MED1            Paracetamol
1   MED2              Vitamin C
2   MED3             Painkiller
3   MED4                Calcium
4   MED5             Antibiotic
5   MED6              BP Tablet
6   MED7                Aspirin
7   MED8             Skin Cream medication_df
  Visit_ID Med_ID
0     V001   MED1
1     V001   MED2
2     V002   MED3
3     V002   MED4
4     V003   MED1
5     V003   MED5
6     V004   MED6
7     V004   MED7
8     V005   MED8
9     V005   MED5 visit_med_df


In [8]:
# EXECUTE SETUP TO INSTANTIATE DATA CONTEXTS

servers_inventory = {

    "Host_ID": ["SRV-01", "SRV-02", "SRV-03"],

    "Role": ["Web Front", "API Gateway", "Database Replica"]

}

live_interfaces = {

    "Interface_ID": ["eth0", "eth1"],

    "Mapped_Host": ["SRV-01", "SRV-02"],

    "IP_Address": ["10.0.0.4", "10.0.0.9"]

}

df_srv = pd.DataFrame(servers_inventory)

df_inf = pd.DataFrame(live_interfaces)

df_srv.to_sql('Servers', conn, index=False, if_exists='replace')

df_inf.to_sql('Interfaces', conn, index=False, if_exists='replace')

# Broken Query Framework

faulty_query = """

               SELECT s.Host_ID, s.Role, i.Interface_ID, i.IP_Address

               FROM Servers s

                        INNER JOIN Interfaces i ON s.Host_ID = i.Mapped_Host; \
 \
               """

print("--- Faulty Output Metrics (Missing Database Replica SRV-03) ---")

pd.read_sql_query(faulty_query, conn)

fixed_query = """

              SELECT s.Host_ID,

                     s.Role,

                     i.Interface_ID,

                     i.IP_Address

              FROM Servers s

                       LEFT JOIN Interfaces i
                                 ON s.Host_ID = i.Mapped_Host; \
 \
              """

print(pd.read_sql_query(fixed_query, conn))

--- Faulty Output Metrics (Missing Database Replica SRV-03) ---
  Host_ID              Role Interface_ID IP_Address
0  SRV-01         Web Front         eth0   10.0.0.4
1  SRV-02       API Gateway         eth1   10.0.0.9
2  SRV-03  Database Replica          NaN        NaN
